# Experiment 1: Document-Level Text Graphs


We compare two graph construction strategies for each document:

1. **Statistical co-occurrence graph**: connects tokens that appear within a fixed sliding window.
2. **Syntactic dependency graph**: connects tokens using spaCy dependency arcs.

Both graph types use the same documents, same tokens, same labels, same vocabulary, and same GAT architecture. The only intended difference is `edge_index`.

In [ ]:
!pip -q install torch_geometric
!pip -q install datasets spacy scikit-learn pandas tqdm kagglehub
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.9 MB/s eta 0:00:00
^C
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Imports and setup

In [ ]:
import re
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
import kagglehub
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from datasets import load_dataset
from scipy.stats import ttest_rel

import spacy

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

nlp = spacy.load("en_core_web_sm", disable=["ner"])
nlp.max_length = 2_000_000

Device: cuda


## 3. Load Dataset 1: Scientific Text Classification

we use 8
 classes

In [ ]:
def load_scientific_text_classification(n_per_class=200, seed=42):
    ds = load_dataset("knowledgator/Scientific-text-classification")

    frames = []
    # Train split
    for split in ds.keys():
        part = pd.DataFrame(ds[split])
        frames.append(part)

    df = pd.concat(frames, ignore_index=True)

    df["text"] = df["text"].astype(str)
    df = df[df["text"].str.strip().str.len() > 100].copy()

    # Pick the most frequent 8 classes
    top_labels = df["label"].value_counts().head(8).index.tolist()
    df = df[df["label"].isin(top_labels)].copy()

    # Remap labels
    old_to_new = {old: new for new, old in enumerate(sorted(top_labels))}
    new_to_old = {new: old for old, new in old_to_new.items()}

    df["original_label"] = df["label"]
    df["label"] = df["label"].map(old_to_new)

    # Balance
    df = (
        df.groupby("label", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))
          .reset_index(drop=True)
    )

    return df, old_to_new, new_to_old

df_scientific, sci_old_to_new, sci_new_to_old = load_scientific_text_classification(n_per_class=200, seed=42)
print("Scientific Text shape:", df_scientific.shape)
print(df_scientific["original_label"].value_counts())
df_scientific.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


arxiv/train.csv:   0%|          | 0.00/54.0M [00:00<?, ?B/s]

pubmed/train.csv:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78631 [00:00<?, ? examples/s]

Scientific Text shape: (1600, 3)
original_label
astrophysics                                  200
computer science                              200
electrical engineering and systems science    200
high energy physics phenomenology             200
high energy physics theory                    200
mathematics                                   200
physics                                       200
quantum physics                               200
Name: count, dtype: int64


/tmp/ipykernel_2695/2139626622.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))


,text,label,original_label
0,We report the detection of phase-locked pola...,0,astrophysics
1,We measured the Rossiter-McLaughlin effect o...,0,astrophysics
2,"With increasing sensitivity, angular resolut...",0,astrophysics
3,We present spectro-polarimetric analysis of ...,0,astrophysics
4,Protoplanetary disks are thought to evolve v...,0,astrophysics


## 4. Load Dataset 2: IMDB Movie Reviews (Kaggle)

this uses the Kaggle IMDb dataset.


In [ ]:
import kagglehub
import os

def load_imdb_kaggle(n_per_class=1000, seed=42):

    path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
    csv_path = os.path.join(path, "IMDB Dataset.csv")

    df = pd.read_csv(csv_path)
    df["text"] = df["review"].astype(str)
    df = df[df["text"].str.strip().str.len() > 100].copy()

    # IMDB has 'sentiment' as 'positive'/'negative'
    old_to_new = {'negative': 0, 'positive': 1}
    new_to_old = {0: 'negative', 1: 'positive'}

    df["original_label"] = df["sentiment"]
    df["label"] = df["sentiment"].map(old_to_new)

    # Balance
    df = (
        df.groupby("label", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))
          .reset_index(drop=True)
    )

    return df, old_to_new, new_to_old

df_imdb, imdb_old_to_new, imdb_new_to_old = load_imdb_kaggle(n_per_class=1000, seed=42)
print("IMDB shape:", df_imdb.shape)
print(df_imdb["original_label"].value_counts())
df_imdb.head()

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
IMDB shape: (2000, 5)
original_label
negative    1000
positive    1000
Name: count, dtype: int64


/tmp/ipykernel_2695/2031831895.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))


,review,sentiment,text,original_label,label
0,This early Pia Zadora vehicle followed a famil...,negative,This early Pia Zadora vehicle followed a famil...,negative,0
1,This movie was alright. Mary-Kate and Ashley p...,negative,This movie was alright. Mary-Kate and Ashley p...,negative,0
2,Alright lets break it down. Why is this one of...,negative,Alright lets break it down. Why is this one of...,negative,0
3,"First off, I'd like to say that the user comme...",negative,"First off, I'd like to say that the user comme...",negative,0
4,"This is the worst film I have ever seen, bar n...",negative,"This is the worst film I have ever seen, bar n...",negative,0


## 5. Inspect examples

In [ ]:
print("Scientific Text example ")
print("Label:", df_scientific.iloc[0]["label"])
print("Original label:", df_scientific.iloc[0]["original_label"])
print(df_scientific.iloc[0]["text"][:1000])


print("IMDB example")
print("Label:", df_imdb.iloc[0]["label"])
print("Original label:", df_imdb.iloc[0]["original_label"])
print(df_imdb.iloc[0]["text"][:1000])

Scientific Text example 
Label: 0
Original label: astrophysics
  We report the detection of phase-locked polarization in the bright ($m_V$=2.98-3.24) semidetached eclipsing binary $\mu^1$ Sco (HD 151890). The phenomenon was observed in multiple photometric bands using two different HIPPI-class (HIgh Precision Polarimetric Instrument)polarimeters with telescopes ranging in size from 35-cm to 3.9-m. The peak-to-trough amplitude of the polarization is wavelength dependent and large, $\sim$700 parts-per-million in green light, and is easily seen with even the smallest telescope. We fit the polarization phase curve with a SYNSPEC/VLIDORT polarized radiative transfer model and a Wilson-Devinney geometric formalism, which we describe in detail. Light from each star reflected by the photosphere of the other, together with a much smaller contribution from tidal distortion and eclipse effects, wholly accounts for the polarization amplitude. In the past polarization in semidetached binaries has b

In [ ]:
active_df = df_scientific.copy()
dataset_name = "ScientificText"



num_classes = active_df["label"].nunique()

print("Using:", dataset_name)
print("Documents:", len(active_df))
print("Classes:", num_classes)
print(active_df["label"].value_counts())

Using: ScientificText
Documents: 1600
Classes: 8
label
0    200
1    200
2    200
3    200
4    200
5    200
6    200
7    200
Name: count, dtype: int64


## 7. Train/validation/test split

In [ ]:
def make_splits(df, seed=42, train_size=0.70, val_size=0.15, test_size=0.15):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6

    texts = df["text"].tolist()
    labels = df["label"].astype(int).tolist()

    train_texts, temp_texts, train_y, temp_y = train_test_split(
        texts,
        labels,
        train_size=train_size,
        stratify=labels,
        random_state=seed
    )

    relative_val_size = val_size / (val_size + test_size)

    val_texts, test_texts, val_y, test_y = train_test_split(
        temp_texts,
        temp_y,
        train_size=relative_val_size,
        stratify=temp_y,
        random_state=seed
    )

    return train_texts, val_texts, test_texts, train_y, val_y, test_y


train_texts, val_texts, test_texts, train_y, val_y, test_y = make_splits(active_df, seed=42)

print("Split sizes:", len(train_texts), len(val_texts), len(test_texts))
print("Train labels:", Counter(train_y))
print("Val labels:", Counter(val_y))
print("Test labels:", Counter(test_y))

Split sizes: 1120 240 240
Train labels: Counter({3: 140, 0: 140, 6: 140, 1: 140, 7: 140, 4: 140, 5: 140, 2: 140})
Val labels: Counter({1: 30, 5: 30, 4: 30, 7: 30, 3: 30, 0: 30, 6: 30, 2: 30})
Test labels: Counter({6: 30, 3: 30, 5: 30, 4: 30, 7: 30, 1: 30, 2: 30, 0: 30})


## 8. Token extraction with spaCy

Each document becomes a list of token dictionaries. We keep the spaCy token index and head index because they are needed for dependency graphs.

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_token_info_batch(texts, max_tokens=256, batch_size=64):
    all_docs = []
    texts = [clean_text(t) for t in texts]

    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts)):
        token_info = []

        for tok in doc:
            if tok.is_space or tok.is_punct:
                continue

            token_info.append({
                "text": tok.text.lower(),
                "i": tok.i,
                "head_i": tok.head.i,
                "dep": tok.dep_,
                "pos": tok.pos_,
                "is_alpha": tok.is_alpha
            })

            if len(token_info) >= max_tokens:
                break

        all_docs.append(token_info)

    return all_docs

In [ ]:
MAX_TOKENS = 256

train_tok = extract_token_info_batch(train_texts, max_tokens=MAX_TOKENS)
val_tok = extract_token_info_batch(val_texts, max_tokens=MAX_TOKENS)
test_tok = extract_token_info_batch(test_texts, max_tokens=MAX_TOKENS)

print("Example tokenized document:")
for i, tok in enumerate(train_tok[0][:20]):
    print(i, tok)

  0%|          | 0/1120 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

Example tokenized document:
0 {'text': 'very', 'i': 0, 'head_i': 1, 'dep': 'advmod', 'pos': 'ADV', 'is_alpha': True}
1 {'text': 'recently', 'i': 1, 'head_i': 12, 'dep': 'advmod', 'pos': 'ADV', 'is_alpha': True}
2 {'text': 'the', 'i': 3, 'head_i': 8, 'dep': 'det', 'pos': 'DET', 'is_alpha': True}
3 {'text': 'muon', 'i': 4, 'head_i': 6, 'dep': 'nmod', 'pos': 'PROPN', 'is_alpha': True}
4 {'text': '$', 'i': 5, 'head_i': 6, 'dep': 'nmod', 'pos': 'SYM', 'is_alpha': False}
5 {'text': 'g-2', 'i': 6, 'head_i': 8, 'dep': 'amod', 'pos': 'NUM', 'is_alpha': False}
6 {'text': '$', 'i': 7, 'head_i': 8, 'dep': 'det', 'pos': 'SYM', 'is_alpha': False}
7 {'text': 'experiment', 'i': 8, 'head_i': 12, 'dep': 'nsubj', 'pos': 'NOUN', 'is_alpha': True}
8 {'text': 'at', 'i': 9, 'head_i': 8, 'dep': 'prep', 'pos': 'ADP', 'is_alpha': True}
9 {'text': 'fermilab', 'i': 10, 'head_i': 9, 'dep': 'pobj', 'pos': 'PROPN', 'is_alpha': True}
10 {'text': 'has', 'i': 11, 'head_i': 12, 'dep': 'aux', 'pos': 'AUX', 'is_alpha': Tr

## 9. Build vocabulary from training split only

In [ ]:
PAD_ID = 0
UNK_ID = 1


def build_vocab(tokenized_docs, min_freq=2, max_vocab=30000):
    counter = Counter()

    for doc in tokenized_docs:
        for tok in doc:
            if tok["is_alpha"]:
                counter[tok["text"]] += 1

    vocab = {
        "<PAD>": PAD_ID,
        "<UNK>": UNK_ID
    }

    for word, freq in counter.most_common(max_vocab - 2):
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab, counter


vocab, word_counter = build_vocab(train_tok, min_freq=2, max_vocab=30000)

print("Vocab size:", len(vocab))
print("Most common:", word_counter.most_common(20))

Vocab size: 7343
Most common: [('the', 12042), ('of', 7008), ('and', 4340), ('a', 4185), ('to', 3847), ('in', 3749), ('we', 2759), ('is', 2134), ('for', 1970), ('that', 1706), ('with', 1676), ('on', 1294), ('this', 1245), ('are', 1134), ('by', 1092), ('as', 1037), ('an', 900), ('be', 827), ('from', 724), ('which', 723)]


## 10. Graph builder A: co-occurrence graph

connect tokens that appear within a fixed sliding window. Reverse edges and self-loops are included.

In [ ]:
def build_cooccurrence_edges(token_info, window_size=3):
    num_nodes = len(token_info)
    edges = set()

    if num_nodes == 0:
        return torch.tensor([[0], [0]], dtype=torch.long)

    for i in range(num_nodes):
        # self-loop
        edges.add((i, i))

        left = max(0, i - window_size)
        right = min(num_nodes, i + window_size + 1)

        for j in range(left, right):
            if i == j:
                continue

            # bidirectional edges
            edges.add((i, j))
            edges.add((j, i))

    edge_index = torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    return edge_index

## 11. Graph builder B: dependency graph

connect each token to its syntactic head according to spaCy. Reverse edges and self-loops are included.

In [ ]:
def build_dependency_edges(token_info):
    num_nodes = len(token_info)
    edges = set()

    if num_nodes == 0:
        return torch.tensor([[0], [0]], dtype=torch.long)

    # spaCy original token index -> node index in our filtered/truncated graph
    original_to_node = {
        tok["i"]: node_i
        for node_i, tok in enumerate(token_info)
    }

    for node_i, tok in enumerate(token_info):
        # self-loop
        edges.add((node_i, node_i))

        head_original_i = tok["head_i"]

        #add edge only if the head token is also kept after filtering
        if head_original_i in original_to_node:
            head_node_i = original_to_node[head_original_i]

            if head_node_i != node_i:
                edges.add((node_i, head_node_i))
                edges.add((head_node_i, node_i))

    edge_index = torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    return edge_index

## 12. Convert tokenized documents to PyTorch Geometric graphs

In [ ]:
def tokens_to_ids(token_info, vocab):
    return [vocab.get(tok["text"], UNK_ID) for tok in token_info]

def make_pyg_graph(token_info, label, vocab, graph_type="cooc", window_size=3):
    if len(token_info) == 0:
        token_ids = [UNK_ID]
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        token_ids = tokens_to_ids(token_info, vocab)

        if graph_type == "cooc":
            edge_index = build_cooccurrence_edges(token_info, window_size=window_size)
        elif graph_type == "dep":
            edge_index = build_dependency_edges(token_info)
        else:
            raise ValueError("graph_type must be 'cooc' or 'dep'")

    x = torch.tensor(token_ids, dtype=torch.long).view(-1, 1)
    y = torch.tensor([label], dtype=torch.long)

    graph = Data(x=x, edge_index=edge_index, y=y)
    graph.num_nodes = x.size(0)

    return graph


def make_graph_dataset(tokenized_docs, labels, vocab, graph_type="cooc", window_size=3):
    graphs = []

    for token_info, label in tqdm(list(zip(tokenized_docs, labels)), total=len(labels)):
        graph = make_pyg_graph(
            token_info=token_info,
            label=label,
            vocab=vocab,
            graph_type=graph_type,
            window_size=window_size
        )
        graphs.append(graph)

    return graphs

## 13. Sanity check: one document, both graph types

In [ ]:
example_idx = 0
example_tokens = train_tok[example_idx]
example_label = train_y[example_idx]

print("Original text:")
print(train_texts[example_idx][:1000])

print("\nTokens:")
for i, tok in enumerate(example_tokens[:40]):
    print(
        f"{i:02d}",
        tok["text"],
        "| spacy_i:", tok["i"],
        "| head_i:", tok["head_i"],
        "| dep:", tok["dep"],
        "| pos:", tok["pos"]
    )

cooc_graph = make_pyg_graph(example_tokens, example_label, vocab, graph_type="cooc", window_size=3)
dep_graph = make_pyg_graph(example_tokens, example_label, vocab, graph_type="dep", window_size=3)

print("\nCo-occurrence graph:")
print(cooc_graph)
print("x shape:", cooc_graph.x.shape)
print("edge_index shape:", cooc_graph.edge_index.shape)
print("label:", cooc_graph.y)

print("\nDependency graph:")
print(dep_graph)
print("x shape:", dep_graph.x.shape)
print("edge_index shape:", dep_graph.edge_index.shape)
print("label:", dep_graph.y)

Original text:
  Very recently, the Muon $g-2$ experiment at Fermilab has confirmed the E821 Brookhaven result, which hinted at a deviation of the muon anomalous magnetic moment from the Standard Model (SM) expectation. The combined results from Brookhaven and Fermilab show a difference with the SM prediction $\delta a_\mu = (251 \pm 59) \times 10^{-11}$ at a significance of $4.2\sigma$, strongly indicating the presence of new physics. Motivated by this new result we reexamine the contributions to the muon anomalous magnetic moment from both: (i)~the ubiquitous $U(1)$ gauge bosons of D-brane string theory constructions and (ii)~the Regge excitations of the string. We show that, for a string scale ${\cal O} ({\rm PeV})$, the contribution from anomalous $U(1)$ gauge bosons which couple to hadrons could help to reduce (though not fully eliminate) the discrepancy reported by the Muon $g-2$ Collaboration. Consistency with null results from LHC searches of new heavy vector bosons imparts the

In [ ]:
def print_edges_readable(token_info, edge_index, max_edges=80):
    id_to_token = {i: tok["text"] for i, tok in enumerate(token_info)}
    edges = edge_index.t().tolist()

    for k, (src, dst) in enumerate(edges[:max_edges]):
        print(
            f"{k:03d}: "
            f"{src:02d}:{id_to_token.get(src, '?')} -> "
            f"{dst:02d}:{id_to_token.get(dst, '?')}"
        )

    if len(edges) > max_edges:
        print(f"... showing {max_edges}/{len(edges)} edges")


print("CO-OCCURRENCE EDGES")
print_edges_readable(example_tokens, cooc_graph.edge_index, max_edges=80)

print("\nDEPENDENCY EDGES")
print_edges_readable(example_tokens, dep_graph.edge_index, max_edges=80)

CO-OCCURRENCE EDGES
000: 18:at -> 17:hinted
001: 29:standard -> 32:expectation
002: 08:at -> 09:fermilab
003: 48:\delta -> 45:sm
004: 40:show -> 41:a
005: 19:a -> 18:at
006: 11:confirmed -> 14:brookhaven
007: 63:indicating -> 61:4.2\sigma$
008: 41:a -> 42:difference
009: 81:anomalous -> 78:to
010: 73:result -> 74:we
011: 52:\pm -> 51:251
012: 44:the -> 47:$
013: 103:the -> 102:of
014: 74:we -> 75:reexamine
015: 114:o -> 111:scale
016: 106:show -> 107:that
017: 85:both -> 84:from
018: 96:theory -> 99:ii)~the
019: 136:eliminate -> 135:fully
020: 107:that -> 108:for
021: 147:consistency -> 144:g-2
022: 118:contribution -> 117:the
023: 129:could -> 132:reduce
024: 148:with -> 145:$
025: 140:by -> 141:the
026: 151:from -> 150:results
027: 184:from -> 184:from
028: 162:constraint -> 165:that
029: 181:comment -> 178:conjectured
030: 173:suppressed -> 174:as
031: 217:boson -> 217:boson
032: 195:\delta -> 198:in
033: 214:the -> 211:kk
034: 206:intersecting -> 207:d
035: 247:fermilab -> 245:broo

## 14. Build full graph datasets

In [ ]:
WINDOW_SIZE = 3

cooc_train_graphs = make_graph_dataset(train_tok, train_y, vocab, graph_type="cooc", window_size=WINDOW_SIZE)
cooc_val_graphs = make_graph_dataset(val_tok, val_y, vocab, graph_type="cooc", window_size=WINDOW_SIZE)
cooc_test_graphs = make_graph_dataset(test_tok, test_y, vocab, graph_type="cooc", window_size=WINDOW_SIZE)

dep_train_graphs = make_graph_dataset(train_tok, train_y, vocab, graph_type="dep", window_size=WINDOW_SIZE)
dep_val_graphs = make_graph_dataset(val_tok, val_y, vocab, graph_type="dep", window_size=WINDOW_SIZE)
dep_test_graphs = make_graph_dataset(test_tok, test_y, vocab, graph_type="dep", window_size=WINDOW_SIZE)

print("Done.")

  0%|          | 0/1120 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/1120 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

Done.


## 15. Graph statistics and validation

Co-occurrence graphs should usually have more edges than dependency graphs.

In [ ]:
def graph_stats(graphs, name):
    nodes = np.array([g.num_nodes for g in graphs])
    edges = np.array([g.edge_index.size(1) for g in graphs])

    print(f"\n{name}")
    print("Num graphs:", len(graphs))
    print("Avg nodes:", nodes.mean())
    print("Median nodes:", np.median(nodes))
    print("Max nodes:", nodes.max())
    print("Avg edges:", edges.mean())
    print("Median edges:", np.median(edges))
    print("Max edges:", edges.max())


graph_stats(cooc_train_graphs, "Co-occurrence train graphs")
graph_stats(dep_train_graphs, "Dependency train graphs")


Co-occurrence train graphs
Num graphs: 1120
Avg nodes: 157.75625
Median nodes: 157.5
Max nodes: 256
Avg edges: 1092.29375
Median edges: 1090.5
Max edges: 1780

Dependency train graphs
Num graphs: 1120
Avg nodes: 157.75625
Median nodes: 157.5
Max nodes: 256
Avg edges: 460.32410714285714
Median edges: 459.5
Max edges: 758


In [ ]:
def validate_graphs(graphs, name):
    for idx, g in enumerate(graphs):
        num_nodes = g.num_nodes

        if g.x.size(0) != num_nodes:
            raise ValueError(f"{name} graph {idx}: x size != num_nodes")

        if g.edge_index.numel() == 0:
            raise ValueError(f"{name} graph {idx}: empty edge_index")

        max_edge = int(g.edge_index.max())
        min_edge = int(g.edge_index.min())

        if min_edge < 0:
            raise ValueError(f"{name} graph {idx}: negative edge index")

        if max_edge >= num_nodes:
            raise ValueError(
                f"{name} graph {idx}: edge index {max_edge} >= num_nodes {num_nodes}"
            )

        if g.y.numel() != 1:
            raise ValueError(f"{name} graph {idx}: y should have one label")

    print(f"{name}: all graphs valid.")


validate_graphs(cooc_train_graphs, "Co-occurrence train")
validate_graphs(dep_train_graphs, "Dependency train")
validate_graphs(cooc_val_graphs, "Co-occurrence val")
validate_graphs(dep_val_graphs, "Dependency val")
validate_graphs(cooc_test_graphs, "Co-occurrence test")
validate_graphs(dep_test_graphs, "Dependency test")

Co-occurrence train: all graphs valid.
Dependency train: all graphs valid.
Co-occurrence val: all graphs valid.
Dependency val: all graphs valid.
Co-occurrence test: all graphs valid.
Dependency test: all graphs valid.


## 16. Create DataLoaders

In [ ]:
BATCH_SIZE = 32

cooc_train_loader = DataLoader(cooc_train_graphs, batch_size=BATCH_SIZE, shuffle=True)
cooc_val_loader = DataLoader(cooc_val_graphs, batch_size=BATCH_SIZE, shuffle=False)
cooc_test_loader = DataLoader(cooc_test_graphs, batch_size=BATCH_SIZE, shuffle=False)

dep_train_loader = DataLoader(dep_train_graphs, batch_size=BATCH_SIZE, shuffle=True)
dep_val_loader = DataLoader(dep_val_graphs, batch_size=BATCH_SIZE, shuffle=False)
dep_test_loader = DataLoader(dep_test_graphs, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(cooc_train_loader))

print(batch)
print("batch.x shape:", batch.x.shape)
print("batch.edge_index shape:", batch.edge_index.shape)
print("batch.y shape:", batch.y.shape)
print("batch.batch shape:", batch.batch.shape)
print("num graphs in batch:", batch.num_graphs)

DataBatch(x=[4507, 1], edge_index=[2, 31165], y=[32], num_nodes=4507, batch=[4507], ptr=[33])
batch.x shape: torch.Size([4507, 1])
batch.edge_index shape: torch.Size([2, 31165])
batch.y shape: torch.Size([32])
batch.batch shape: torch.Size([4507])
num graphs in batch: 32


## 17. Define the GAT model

Both graph conditions use the same model architecture.

In [ ]:
class TextGraphGAT(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        emb_dim=128,
        hidden_dim=64,
        heads=4,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            padding_idx=PAD_ID
        )

        self.gat1 = GATConv(
            in_channels=emb_dim,
            out_channels=hidden_dim,
            heads=heads,
            concat=True,
            dropout=dropout,
            add_self_loops=False
        )

        self.gat2 = GATConv(
            in_channels=hidden_dim * heads,
            out_channels=hidden_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            add_self_loops=False
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, batch):
        token_ids = batch.x.squeeze(-1)

        x = self.embedding(token_ids)

        x = self.gat1(x, batch.edge_index)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat2(x, batch.edge_index)
        x = F.elu(x)

        graph_emb = global_mean_pool(x, batch.batch)
        logits = self.classifier(self.dropout(graph_emb))

        return logits

## 18. Instantiate models and run a forward-pass check

In [ ]:
vocab_size = len(vocab)
num_classes = active_df["label"].nunique()

cooc_model = TextGraphGAT(
    vocab_size=vocab_size,
    num_classes=num_classes,
    emb_dim=128,
    hidden_dim=64,
    heads=4,
    dropout=0.3
).to(device)

dep_model = TextGraphGAT(
    vocab_size=vocab_size,
    num_classes=num_classes,
    emb_dim=128,
    hidden_dim=64,
    heads=4,
    dropout=0.3
).to(device)

print(cooc_model)

TextGraphGAT(
  (embedding): Embedding(7343, 128, padding_idx=0)
  (gat1): GATConv(128, 64, heads=4)
  (gat2): GATConv(256, 64, heads=1)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=64, out_features=8, bias=True)
)


In [ ]:
cooc_batch = next(iter(cooc_train_loader)).to(device)
dep_batch = next(iter(dep_train_loader)).to(device)

cooc_model.eval()
dep_model.eval()

with torch.no_grad():
    cooc_logits = cooc_model(cooc_batch)
    dep_logits = dep_model(dep_batch)

print("Co-occurrence logits shape:", cooc_logits.shape)
print("Dependency logits shape:", dep_logits.shape)
print("Expected:", cooc_batch.num_graphs, "x", num_classes)

criterion = nn.CrossEntropyLoss()
cooc_loss = criterion(cooc_logits, cooc_batch.y.view(-1))
dep_loss = criterion(dep_logits, dep_batch.y.view(-1))

print("Co-occurrence dummy loss:", cooc_loss.item())
print("Dependency dummy loss:", dep_loss.item())

if torch.cuda.is_available():
    print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
    print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)

Co-occurrence logits shape: torch.Size([32, 8])
Dependency logits shape: torch.Size([32, 8])
Expected: 32 x 8
Co-occurrence dummy loss: 2.0618886947631836
Dependency dummy loss: 2.091902256011963
Allocated GB: 0.01722240447998047
Reserved GB: 0.115234375


## 19. Final sanity summary

If logits have shape `[batch_size, num_classes]` and losses are finite, the pipeline works up to model construction.

In [ ]:
print("check")
print("Dataset:", dataset_name)
print("Documents:", len(active_df))
print("Classes:", num_classes)
print("Vocab size:", vocab_size)

print("\nSplits:")
print("Train:", len(train_y))
print("Val:", len(val_y))
print("Test:", len(test_y))

print("\nGraph datasets:")
print("Cooc train graphs:", len(cooc_train_graphs))
print("Dep train graphs:", len(dep_train_graphs))

print("\nOne cooc graph:")
print(cooc_train_graphs[0])

print("\nOne dep graph:")
print(dep_train_graphs[0])

print("\nForward pass:")
print("Cooc logits:", cooc_logits.shape)
print("Dep logits:", dep_logits.shape)

check
Dataset: ScientificText
Documents: 1600
Classes: 8
Vocab size: 7343

Splits:
Train: 1120
Val: 240
Test: 240

Graph datasets:
Cooc train graphs: 1120
Dep train graphs: 1120

One cooc graph:
Data(x=[248, 1], edge_index=[2, 1724], y=[1], num_nodes=248)

One dep graph:
Data(x=[248, 1], edge_index=[2, 728], y=[1], num_nodes=248)

Forward pass:
Cooc logits: torch.Size([32, 8])
Dep logits: torch.Size([32, 8])


```markdown
We have successfully built the complete graph construction and training pipeline for Experiment 1. We preprocessed the texts with spaCy and built a train-only vocabulary to represent tokens as numerical nodes, which allowed us to generate two parallel PyTorch Geometric graph structures per document: co-occurrence graphs (sliding window edges) and dependency graphs (syntactic head edges). After formatting these into batch-ready DataLoaders, we defined a standard GAT model (Embedding, GATs, Mean Pooling, Classifier) and validated it with a successful forward-pass sanity check.
```

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        logits = model(batch)
        loss = F.cross_entropy(logits, batch.y.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_full(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)

        logits = model(batch)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch.y.view(-1).cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return acc, macro_f1, all_preds, all_labels

In [ ]:
import copy

def run_experiment(
    train_loader, val_loader, test_loader,
    vocab_size, num_classes,
    seeds=[13, 21, 42, 87, 100],
    patience=5, max_epochs=30
):
    all_test_accs = []
    all_test_f1s = []

    last_seed_preds = None
    last_seed_labels = None

    for seed in seeds:
        set_seed(seed)

        model = TextGraphGAT(
            vocab_size=vocab_size,
            num_classes=num_classes,
            emb_dim=128,
            hidden_dim=64,
            heads=4,
            dropout=0.3
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_f1 = -1
        patience_counter = 0
        best_model_state = None

        for epoch in range(1, max_epochs + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer)
            val_acc, val_f1, _, _ = evaluate_full(model, val_loader)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1

            if patience_counter >= patience:
                break

        # Load best model for testing
        if best_model_state is not None:
            model.load_state_dict(best_model_state)

        test_acc, test_f1, curr_best_preds, curr_best_labels = evaluate_full(model, test_loader)

        all_test_accs.append(test_acc)
        all_test_f1s.append(test_f1)

        # Save predictions from the last seed run for CM/ClassReport
        last_seed_preds = curr_best_preds
        last_seed_labels = curr_best_labels

    acc_mean, acc_std = np.mean(all_test_accs), np.std(all_test_accs, ddof=1)
    f1_mean, f1_std = np.mean(all_test_f1s), np.std(all_test_f1s, ddof=1)

    return {
        "acc_mean": acc_mean * 100, "acc_std": acc_std * 100,
        "f1_mean": f1_mean * 100, "f1_std": f1_std * 100,
        "all_accs": all_test_accs, "all_f1s": all_test_f1s,
        "preds": last_seed_preds, "labels": last_seed_labels
    }

In [ ]:
def process_spacy_doc(doc, max_tokens=256):
    token_info = []

    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue

        token_info.append({
            "text": tok.text.lower(),
            "i": tok.i,
            "head_i": tok.head.i,
            "dep": tok.dep_,
            "pos": tok.pos_,
            "is_alpha": tok.is_alpha
        })

        if len(token_info) >= max_tokens:
            break

    return token_info

In [ ]:
def run_experiment_pipeline(df, dataset_name, label_names_dict):
    print(f"running for: {dataset_name}")

    # 1. Splits
    train_texts, val_texts, test_texts, train_y, val_y, test_y = make_splits(df, seed=42)

    # 2. NLP Processing
    print(f"[{dataset_name}] processing with spacy")
    train_tok = []
    for doc in tqdm(nlp.pipe(train_texts, batch_size=128), total=len(train_texts)):
        train_tok.append(process_spacy_doc(doc))
    val_tok = []
    for doc in tqdm(nlp.pipe(val_texts, batch_size=128), total=len(val_texts)):
        val_tok.append(process_spacy_doc(doc))
    test_tok = []
    for doc in tqdm(nlp.pipe(test_texts, batch_size=128), total=len(test_texts)):
        test_tok.append(process_spacy_doc(doc))

    # 3. Vocab
    vocab, _ = build_vocab(train_tok, min_freq=2, max_vocab=30000)
    vocab_size = len(vocab)
    num_classes = df["label"].nunique()
    print(f"[{dataset_name}] Vocab size: {vocab_size}, Classes: {num_classes}")

    # 4. Graph Construction
    WINDOW_SIZE = 3
    print(f"[{dataset_name}] building statistic graph")
    cooc_train_graphs = make_graph_dataset(train_tok, train_y, vocab, graph_type="cooc", window_size=WINDOW_SIZE)
    cooc_val_graphs = make_graph_dataset(val_tok, val_y, vocab, graph_type="cooc", window_size=WINDOW_SIZE)
    cooc_test_graphs = make_graph_dataset(test_tok, test_y, vocab, graph_type="cooc", window_size=WINDOW_SIZE)

    print(f"[{dataset_name}] building syntactic graph")
    dep_train_graphs = make_graph_dataset(train_tok, train_y, vocab, graph_type="dep", window_size=WINDOW_SIZE)
    dep_val_graphs = make_graph_dataset(val_tok, val_y, vocab, graph_type="dep", window_size=WINDOW_SIZE)
    dep_test_graphs = make_graph_dataset(test_tok, test_y, vocab, graph_type="dep", window_size=WINDOW_SIZE)

    # Graph Stats
    graph_stats(cooc_train_graphs, f"{dataset_name} co-occurrence Train")
    graph_stats(dep_train_graphs, f"{dataset_name} dependency Train")

    # Edge Density Ratio
    cooc_avg_edges = np.mean([g.edge_index.size(1) for g in cooc_train_graphs])
    dep_avg_edges = np.mean([g.edge_index.size(1) for g in dep_train_graphs])
    print(f"[{dataset_name}] Edge density ratio cooc/dep: {cooc_avg_edges / dep_avg_edges:.2f}")

    # DataLoader
    BATCH_SIZE = 32
    cooc_train_loader = DataLoader(cooc_train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    cooc_val_loader   = DataLoader(cooc_val_graphs, batch_size=BATCH_SIZE, shuffle=False)
    cooc_test_loader  = DataLoader(cooc_test_graphs, batch_size=BATCH_SIZE, shuffle=False)

    dep_train_loader = DataLoader(dep_train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    dep_val_loader   = DataLoader(dep_val_graphs, batch_size=BATCH_SIZE, shuffle=False)
    dep_test_loader  = DataLoader(dep_test_graphs, batch_size=BATCH_SIZE, shuffle=False)

    # 5. Training and Evaluation Pipeline
    print(f"\n[{dataset_name}] starting Training: Co-occurrence GAT")
    res_cooc = run_experiment(cooc_train_loader, cooc_val_loader, cooc_test_loader, vocab_size, num_classes)

    print(f"[{dataset_name}] starting Training: Dependency GAT")
    res_dep = run_experiment(dep_train_loader, dep_val_loader, dep_test_loader, vocab_size, num_classes)

    # 6. Statistical Test
    _, p_val_acc = ttest_rel(res_cooc["all_accs"], res_dep["all_accs"])
    _, p_val_f1 = ttest_rel(res_cooc["all_f1s"], res_dep["all_f1s"])

    print(f"\n[{dataset_name}] results from 5 seeds")
    print(f"Co-occurrence  Acc: {res_cooc['acc_mean']:.2f} ± {res_cooc['acc_std']:.2f}% | F1: {res_cooc['f1_mean']:.2f} ± {res_cooc['f1_std']:.2f}%")
    print(f"Dependency     Acc: {res_dep['acc_mean']:.2f} ± {res_dep['acc_std']:.2f}% | F1: {res_dep['f1_mean']:.2f} ± {res_dep['f1_std']:.2f}%")
    print(f"Significance (p-value, paired t-test)  Acc: {p_val_acc:.4f} | F1: {p_val_f1:.4f}")

    target_names = [str(label_names_dict.get(i, i)) for i in range(num_classes)]
    print(f"\n[{dataset_name}] statistic/co.ocurrence Classification Report, last seed")
    print(classification_report(res_cooc["labels"], res_cooc["preds"], target_names=target_names, zero_division=0))
    print(f"\n[{dataset_name}] syntactic/dependency GAT Classification Report, last seed")
    print(classification_report(res_dep["labels"], res_dep["preds"], target_names=target_names, zero_division=0))

    return res_cooc, res_dep

In [ ]:
# execution 1: Scientific Text Classification
res_sci_cooc, res_sci_dep = run_experiment_pipeline(df_scientific, "Scientific Text", sci_new_to_old)

# execution 2: IMDb Movie Reviews
res_imdb_cooc, res_imdb_dep = run_experiment_pipeline(df_imdb, "IMDb Movie Reviews", imdb_new_to_old)

running for: Scientific Text
[Scientific Text] processing with spacy


  0%|          | 0/1120 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

[Scientific Text] Vocab size: 7343, Classes: 8
[Scientific Text] building statistic graph


...

[IMDb Movie Reviews] building syntactic graph


  0%|          | 0/1400 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]


IMDb Movie Reviews co-occurrence Train
Num graphs: 1400
Avg nodes: 180.705
Median nodes: 178.0
Max nodes: 256
Avg edges: 1252.935
Median edges: 1234.0
Max edges: 1780

IMDb Movie Reviews dependency Train
Num graphs: 1400
Avg nodes: 180.705
Median nodes: 178.0
Max nodes: 256
Avg edges: 522.1092857142858
Median edges: 514.0
Max edges: 762
[IMDb Movie Reviews] Edge density ratio cooc/dep: 2.40

[IMDb Movie Reviews] starting Training: Co-occurrence GAT
[IMDb Movie Reviews] starting Training: Dependency GAT

[IMDb Movie Reviews] results from 5 seeds
Co-occurrence  Acc: 74.60 ± 1.67% | F1: 74.43 ± 1.72%
Dependency     Acc: 76.47 ± 1.12% | F1: 76.40 ± 1.16%
Significance (p-value, paired t-test)  Acc: 0.1719 | F1: 0.1679

[IMDb Movie Reviews] statistic/co.ocurrence Classification Report, last seed
              precision    recall  f1-score   support

    negative       0.84      0.66      0.74       150
    positive       0.72      0.87      0.79       150

    accuracy                      